# 📊 Merging Dynamic Events and Tracking Data Tutorial

## 📋 Step 1: Setup & Prerequisites

In [29]:

default_palette=[
    "#252525",
    "#00a82f", #  Green
    "#32fe6b", #  Lime
    "#7acbff", #  Pastel azure
    "#2388c8", #  Azure
    "#064c62"  #  Dark Azure
]
import os
from io import BytesIO
import pandas as pd
import json
import numpy as np


# Setup pitch and plot
from mplsoccer.pitch import Pitch ,VerticalPitch

username = os.environ.get("SKILLCORNER_USERNAME")
password = os.environ.get("SKILLCORNER_PASSWORD")


from skillcorner.client import SkillcornerClient
client=SkillcornerClient(username=username,password=password)




def time_to_seconds(time_str):
    if time_str is None:
        return 90 * 60  # 120 minutes = 7200 seconds
    h, m, s = map(int, time_str.split(':'))
    return h * 3600 + m * 60 + s




## 📥 Step 2: Load Tracking Data

In [22]:
####-----------------------------------------------------------------------------------
# If you're a skillcorner client and you know your match_id you can use the pyton client
###------------------------------------------------------------------------------------
# match_id=1886347
# client.save_match_tracking_data(match_id,
#                                 params={'data_version':3},
#                                 filepath=f'{match_id}_tracking_raw.jsonl')
# raw_data=pd.read_json(f'{match_id}_tracking_raw.jsonl')
# os.remove(f'{match_id}_tracking_raw.jsonl')
####-----------------------------------------------------------------------------------
# If you've cloned the repo and are in your local copy
###------------------------------------------------------------------------------------
match_id = 1886347
raw_data = pd.read_json(
    f"../../../data/matches/{match_id}/{match_id}_tracking_extrapolated.jsonl", lines=True
)

####-----------------------------------------------------------------------------------
# If you're on a separate project/environemnt
###------------------------------------------------------------------------------------

# # Construct the raw GitHub URL
# tracking_data_github_url=f'https://media.githubusercontent.com/media/SkillCorner/opendata/741bdb798b0c1835057e3fa77244c1571a00e4aa/data/matches/{match_id}/{match_id}_tracking_extrapolated.jsonl' # Data is stored using GitLFS
# raw_data=pd.read_json(tracking_data_github_url,lines=True)


raw_df = pd.json_normalize(
    raw_data.to_dict("records"),
    "player_data",
    ["frame", "timestamp", "period", "possession", "ball_data"],
)

# Extract 'player_id' and 'group from the 'possession' dictionary
raw_df["possession_player_id"] = raw_df["possession"].apply(
    lambda x: x.get("player_id")
)
raw_df["possession_group"] = raw_df["possession"].apply(lambda x: x.get("group"))


# (Optional) Expand the ball_data with json_normalize
raw_df[["ball_x", "ball_y", "ball_z", "is_detected_ball"]] = pd.json_normalize(
    raw_df.ball_data
)


# (Optional) Drop the original 'possession' column if you no longer need it
raw_df = raw_df.drop(columns=["possession", "ball_data"])

# Add the match_id identifier to your dataframe
raw_df["match_id"] = match_id
tracking_df = raw_df.copy()
tracking_df.head()

,x,y,player_id,is_detected,frame,timestamp,period,possession_player_id,possession_group,ball_x,ball_y,ball_z,is_detected_ball,match_id
0,-39.63,-0.08,51009,False,10,2026-03-31,1.0,NaN,None,0.32,0.38,0.13,True,1886347
1,-19.21,-9.18,176224,True,10,2026-03-31,1.0,NaN,None,0.32,0.38,0.13,True,1886347
2,-21.83,0.47,51649,True,10,2026-03-31,1.0,NaN,None,0.32,0.38,0.13,True,1886347
3,-1.16,-32.47,50983,True,10,2026-03-31,1.0,NaN,None,0.32,0.38,0.13,True,1886347
4,-18.88,15.73,735578,True,10,2026-03-31,1.0,NaN,None,0.32,0.38,0.13,True,1886347


We have now loaded the tracking data

## 📂 Step 3: Load Match Metadata

In [23]:
####-----------------------------------------------------------------------------------
# If you're a skillcorner client and you know your match_id you can use the python client
###------------------------------------------------------------------------------------
# match_id=1886347
# raw_match_data=client.get_match(match_id)


####-----------------------------------------------------------------------------------
# If you've cloned the repo and are in your local copy
###------------------------------------------------------------------------------------
match_id = 1886347
file_path = f"../../../data/matches/{match_id}/{match_id}_match.json"

with open(file_path, "r") as f:
    raw_match_data = json.load(f)

####-----------------------------------------------------------------------------------
# If you're on a separate project/environemnt
###------------------------------------------------------------------------------------
# match_id=1886347
# meta_data_github_url=f'https://raw.githubusercontent.com/SkillCorner/opendata/741bdb798b0c1835057e3fa77244c1571a00e4aa/data/matches/{match_id}/{match_id}_match.json'
# # Read the JSON data as a JSON object
# response = requests.get(meta_data_github_url)
# raw_match_data = response.json()


# The output has nested json elements. We process them
raw_match_df = pd.json_normalize(raw_match_data, max_level=2)
raw_match_df["home_team_side"] = raw_match_df["home_team_side"].astype(str)

players_df = pd.json_normalize(
    raw_match_df.to_dict("records"),
    record_path="players",
    meta=[
        "home_team_score",
        "away_team_score",
        "date_time",
        "home_team_side",
        "home_team.name",
        "home_team.id",
        "away_team.name",
        "away_team.id",
    ],  # data we keep
)


# Take only players who played and create their total time
players_df = players_df[
    ~((players_df.start_time.isna()) & (players_df.end_time.isna()))
]
players_df["total_time"] = players_df["end_time"].apply(time_to_seconds) - players_df[
    "start_time"
].apply(time_to_seconds)

# Create a flag for GK
players_df["is_gk"] = players_df["player_role.acronym"] == "GK"

# Add a flag if the given player is home or away
players_df["match_name"] = (
    players_df["home_team.name"] + " vs " + players_df["away_team.name"]
)


# Add a flag if the given player is home or away
players_df["home_away_player"] = np.where(
    players_df.team_id == players_df["home_team.id"], "Home", "Away"
)

# Create flag from player
players_df["team_name"] = np.where(
    players_df.team_id == players_df["home_team.id"],
    players_df["home_team.name"],
    players_df["away_team.name"],
)

# Figure out sides
players_df[["home_team_side_1st_half", "home_team_side_2nd_half"]] = (
    players_df["home_team_side"]
    .astype(str)
    .str.strip("[]")
    .str.replace("'", "")
    .str.split(", ", expand=True)
)
# Clean up sides
players_df["direction_player_1st_half"] = np.where(
    players_df.home_away_player == "Home",
    players_df.home_team_side_1st_half,
    players_df.home_team_side_2nd_half,
)
players_df["direction_player_2nd_half"] = np.where(
    players_df.home_away_player == "Home",
    players_df.home_team_side_2nd_half,
    players_df.home_team_side_1st_half,
)


# Clean up and keep the columns that we want to keep about

columns_to_keep = [
    "start_time",
    "end_time",
    "match_name",
    "date_time",
    "home_team.name",
    "away_team.name",
    "id",
    "short_name",
    "number",
    "team_id",
    "team_name",
    "player_role.position_group",
    "total_time",
    "player_role.name",
    "player_role.acronym",
    "is_gk",
    "direction_player_1st_half",
    "direction_player_2nd_half",
    "home_away_player"
]
players_df = players_df[columns_to_keep]
players_df.head()

,start_time,end_time,match_name,date_time,home_team.name,away_team.name,id,short_name,number,team_id,team_name,player_role.position_group,total_time,player_role.name,player_role.acronym,is_gk,direction_player_1st_half,direction_player_2nd_half,home_away_player
0,00:00:00,01:25:21,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,38673,G. May,10,4177,Auckland FC,Center Forward,5121,Center Forward,CF,False,right_to_left,left_to_right,Home
1,00:00:00,None,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,51713,C. Elliott,17,4177,Auckland FC,Full Back,5400,Right Back,RB,False,right_to_left,left_to_right,Home
2,00:00:00,01:16:37,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,50951,J. Brimmer,22,4177,Auckland FC,Center Forward,4597,Center Forward,CF,False,right_to_left,left_to_right,Home
3,00:00:00,01:24:58,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,50978,C. Timmins,19,1805,Newcastle United Jets FC,Midfield,5098,Left Defensive Midfield,LDM,False,left_to_right,right_to_left,Away
4,00:00:00,None,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,133498,F. De Vries,15,4177,Auckland FC,Full Back,5400,Left Back,LB,False,right_to_left,left_to_right,Home


Metadata is now ready !

## 🔗 Step 4: Merge Dataframes

In [44]:
# 1. Merge tracking and player metadata
temp_df = tracking_df.merge(players_df, left_on="player_id", right_on="id")

# 2. Assign direction based on the current period
# Uses the period as an index to pick between 1st and 2nd half columns
temp_df['current_direction'] = np.where(
    temp_df['period'] == 1,
    temp_df['direction_player_1st_half'],
    temp_df['direction_player_2nd_half']
)

# 3. Determine possession status
# We simplify the boolean check by matching the team labels
is_home_poss = (temp_df['possession_group'] == 'home team') & (temp_df['home_away_player'] == 'Home')
is_away_poss = (temp_df['possession_group'] == 'away team') & (temp_df['home_away_player'] == 'Away')
temp_df['in_possession'] = is_home_poss | is_away_poss

# 4. Create the Flip Mask
# We flip if:
# (Attacking right-to-left AND in possession) OR (Defending left-to-right AND out of possession)
flip_mask = (
    ((temp_df['current_direction'] == 'right_to_left') & temp_df['in_possession']) |
    ((temp_df['current_direction'] == 'left_to_right') & ~temp_df['in_possession'])
)

# 5. Apply coordinate inversion
# Iterating over existing columns avoids repetitive 'np.where' blocks
coords = [c for c in ['x', 'y', 'ball_x', 'ball_y'] if c in temp_df.columns]
for col in coords:
    temp_df[col] = np.where(flip_mask, -temp_df[col], temp_df[col])

enriched_tracking_data = temp_df


Our tracking data is now ready !

---

# 4.0 READ DYNAMIC EVENT FILE AND ISOLATE ONE EVENT

In [45]:
match_id = 1886347
# If you're using OpenSource Data
de_match = pd.read_csv(f"../../../data/matches/{match_id}/{match_id}_dynamic_events.csv")

# If you're using the client
# de_match=client.get_dynamic_events(match_id)
# de_match= pd.read_csv(BytesIO(de_match))

de_match.head()

,event_id,index,match_id,frame_start,frame_end,frame_physical_start,time_start,time_end,minute_start,second_start,...,xloss_player_possession_end,xloss_player_possession_max,xshot_player_possession_start,xshot_player_possession_end,xshot_player_possession_max,is_player_possession_start_matched,is_player_possession_end_matched,is_previous_pass_matched,is_pass_reception_matched,fully_extrapolated
0,8_0,0,1886347,28,28,NaN,00:01.8,00:01.8,0,1,...,NaN,NaN,NaN,NaN,NaN,True,True,NaN,True,False
1,8_1,1,1886347,48,58,NaN,00:03.8,00:04.8,0,3,...,NaN,NaN,NaN,NaN,NaN,True,True,True,True,False
2,7_0,2,1886347,48,53,NaN,00:03.8,00:04.3,0,3,...,NaN,NaN,NaN,NaN,NaN,True,True,NaN,NaN,False
3,7_1,3,1886347,48,58,NaN,00:03.8,00:04.8,0,3,...,NaN,NaN,NaN,NaN,NaN,True,True,NaN,True,False
4,9_0,4,1886347,56,58,34.0,00:02.4,00:04.8,0,2,...,NaN,NaN,NaN,NaN,NaN,True,True,True,True,NaN


In [46]:
# This is a run in behind
de_match[de_match.event_id == "1_2"][
    [
        "player_id",
        "frame_start",
        "frame_end",
        "event_type",
        "event_subtype",
        "team_id",
        "player_in_possession_id",
        "x_start",
        "y_start",
        "x_end",
        "y_end",
    ]
]

,player_id,frame_start,frame_end,event_type,event_subtype,team_id,player_in_possession_id,x_start,y_start,x_end,y_end
13,966120,82,99,off_ball_run,behind,1805,735578.0,15.51,6.38,19.96,13.36


# 5.0 MERGE TRACKING DATA OVER EVENT DURATION AND ANIMATE

In [48]:
# 5.0 ANIMATE OFF-BALL RUN
# Select a specific run to animate
event_id = "1_133"
specific_event = de_match[de_match.event_id == event_id].iloc[0]

# Get frame range
start_frame = specific_event['frame_start']
end_frame = specific_event['frame_end']

# Filter tracking data
event_tracking_data = enriched_tracking_data[
    (enriched_tracking_data['frame'] >= start_frame) & 
    (enriched_tracking_data['frame'] <= end_frame)
].copy()

# Sync metadata
event_tracking_data['runner'] = specific_event['player_id'] == event_tracking_data['player_id']
event_tracking_data['ball_carrier'] = specific_event['player_in_possession_id'] == event_tracking_data['player_id']
event_tracking_data['tip'] = specific_event['team_id'] == event_tracking_data['team_id']


# --- Animation Logic ---
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

frames = sorted(event_tracking_data['frame'].unique())

pitch = Pitch(
    pitch_type="skillcorner",
    line_alpha=0.75,
    pitch_length=105,
    pitch_width=68,
    pitch_color=default_palette[0],
    line_color='white',
    linewidth=1.5,
)
fig, ax = pitch.grid(figheight=8, endnote_height=0, title_height=0)

# FORCE LIMITS to ensure coordinate consistency
ax.set_xlim(-52.5, 52.5)
ax.set_ylim(-34, 34)

size = 300

# Get the actual starting position from the tracking data to ensure perfect alignment
runner_start_row = event_tracking_data[(event_tracking_data['frame'] == start_frame) & 
                                         (event_tracking_data['player_id'] == specific_event['player_id'])]

if not runner_start_row.empty:
    runner_start_x = runner_start_row.iloc[0]['x']
    runner_start_y = runner_start_row.iloc[0]['y']
else:
    runner_start_x = specific_event['x_start']
    runner_start_y = specific_event['y_start']


# Initialize plots
# Use consistent colors: Team in possession is always palette[1]
possession_scatter = ax.scatter([], [], c=default_palette[1], alpha=0.95, s=size, edgecolors=default_palette[1], linewidths=1.5, zorder=10, label="Team In Possession")
out_possession_scatter = ax.scatter([], [], c=default_palette[4], alpha=0.95, s=size, edgecolors=default_palette[4], linewidths=1, zorder=10, label="Team Out of Possession")
runner_scatter = ax.scatter([], [], c=default_palette[1], alpha=0.95, s=size, edgecolors="white", linewidths=2, zorder=10, label="Runner")
runner_start_scatter = ax.scatter([runner_start_x], [runner_start_y], c=default_palette[1], alpha=0.55, s=size/2, edgecolors=default_palette[1], linewidths=2, zorder=10, label="Run Origin")
runner_line, = ax.plot([], [], color=default_palette[1], linewidth=2, ls="--")
ball_carrier_scatter = ax.scatter([], [], c=default_palette[2], alpha=0.95, s=size, edgecolors=default_palette[2], linewidths=2.5, zorder=10, label="Ball Carrier")
ball_scatter = ax.scatter([], [], c="white", alpha=0.95, s=size/3, edgecolors="black", linewidths=1.5, zorder=15, label="Ball")

pitch.draw(ax=ax)

def update(frame):
    current_data = event_tracking_data[event_tracking_data['frame'] == frame]
    
    # Update positions (tracking data is now synced to match CSV)
    pos_data = current_data[current_data['tip'] == True]
    if len(pos_data) > 0:
        possession_scatter.set_offsets(pos_data[['x', 'y']].values)
    else:
        possession_scatter.set_offsets(np.empty((0, 2)))
        
    out_pos_data = current_data[current_data['tip'] == False]
    if len(out_pos_data) > 0:
        out_possession_scatter.set_offsets(out_pos_data[['x', 'y']].values)
    else:
        out_possession_scatter.set_offsets(np.empty((0, 2)))
        
    run_data = current_data[current_data['runner'] == True]
    if len(run_data) > 0:
        runner_scatter.set_offsets(run_data[['x', 'y']].values)
        # Connect start marker (CSV) to current position (Tracking)
        runner_line.set_data([runner_start_x, run_data['x'].values[0]], [runner_start_y, run_data['y'].values[0]])
    else:
        runner_scatter.set_offsets(np.empty((0, 2)))
        runner_line.set_data([], [])
        
    bc_data = current_data[current_data['ball_carrier'] == True]
    if len(bc_data) > 0:
        ball_carrier_scatter.set_offsets(bc_data[['x', 'y']].values)
    else:
        ball_carrier_scatter.set_offsets(np.empty((0, 2)))
        
    if len(current_data) > 0 and pd.notnull(current_data['ball_x'].iloc[0]):
        ball_scatter.set_offsets(np.array([[current_data['ball_x'].iloc[0], current_data['ball_y'].iloc[0]]]))
    else:
        ball_scatter.set_offsets(np.empty((0, 2)))

    return possession_scatter, out_possession_scatter, runner_scatter, runner_line, ball_carrier_scatter, ball_scatter

anim = FuncAnimation(fig, update, frames=frames, interval=100, blit=True)
import matplotlib.pyplot as plt
plt.close(fig)
print(f"Animation ready for {event_id} (Synced: {len(frames)} frames)")
HTML(anim.to_jshtml())

Animation ready for 1_133 (Synced: 77 frames)
